# **Purpose**

The purpose of this notebook is to evaluate the fine-tuned `llama-3.2-3b-instruct` model. It scores the model on a **fixed SQuAD test split**, so results are directly comparable across all the models (Flan-T5, BART, GPT-2, TinyLlama, Qwen2.5, Llama-3.2, and Phi-3.5).

**Metrics:** BLEU, ROUGE-1/2/L, METEOR, GPT-2 perplexity (fluency), Distinct-1/2 (diversity).

**How to reuse across models:** change `MODEL_REPO` (and `MODEL_TYPE` if switching between seq2seq and causal-LM models) in the config cell, then rerun the whole notebook. Keep `TEST_SIZE` and `SEED` identical across every run so all models are scored on the exact same examples.


## **Install dependencies**

In [1]:
!pip install -qU \
 transformers \
 evaluate \
 rouge_score \
 sacrebleu \
 nltk \
 torch \
 torchvision \
 torchaudio \
 bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 98.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 66.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.6/526.6 MB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.7/197.7 MB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 105.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90

In [2]:
import os
os.environ["NLTK_ALLOW_PROXIED_URLOPEN"] = "1"

import nltk
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

True

## **Imports**

In [3]:
import json
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    AutoModelForCausalLM,
    GPT2LMHeadModel,
    GPT2TokenizerFast,
    GPT2Tokenizer,
)
import evaluate

from kaggle_secrets import UserSecretsClient

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

Device: cuda


## **Load the API Keys and Tokens**

In [4]:
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")   # must match the exact secret name we set in Kaggle Secrets

os.environ["HF_TOKEN"] = hf_token

## **Config**

- `MODEL_REPO`: the Hugging Face Hub repo of the fine-tuned model we want to evaluate.
- `MODEL_TYPE`: `"seq2seq"` for T5/BART/Flan-T5, `"causal"` for Qwen/Llama/Phi.
- `TEST_SIZE` / `SEED`: **We need to keep these identical across every model we evaluate** - this is what makes the comparison fair.

In [5]:
MODEL_REPO = "gaurav-dey/llama3.2-3b-qg"   # change this per model
MODEL_TYPE = "causal"                          # "seq2seq" or "causal"

TEST_SIZE = 1500      # keep identical across all models you compare
SEED = 42            # keep identical across all models you compare

NUM_BEAMS = 4
MAX_INPUT_LENGTH = 512   # bumped from 384
MAX_TARGET_LENGTH = 96

OUTPUT_JSON = f"/kaggle/working/eval_results_{MODEL_REPO.split('/')[-1]}.json"

SYSTEM_PROMPT = (
    "You are a question generation assistant. Given a context passage and a target answer, "
    "generate a single question whose correct answer is exactly the target answer."
)   # paste the exact same system prompt used during Llama fine-tuning

## **Load the fixed test split**

Shuffles SQuAD validation with a fixed seed and takes the first `TEST_SIZE` examples. As long as `TEST_SIZE`/`SEED` don't change between runs, every model we evaluate sees the exact same examples.

In [6]:
raw = load_dataset("squad")
val_ds = raw["validation"].shuffle(seed=SEED)
test_ds = val_ds.select(range(TEST_SIZE))

print(test_ds)

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 1500
})


## **Load the model and tokenizer from the Hub**

In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_REPO)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if MODEL_TYPE == "seq2seq":
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_REPO).to(device)
else:
    model = AutoModelForCausalLM.from_pretrained(MODEL_REPO).to(device)

model.eval()
print(f"Loaded {MODEL_REPO} ({MODEL_TYPE})")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/354 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.03G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/222 [00:00<?, ?B/s]

Loaded gaurav-dey/llama3.2-3b-qg (causal)


## **Prompt builder**

Must match whatever format the model was fine-tuned with (answer + instruction placed BEFORE the context, so truncation only ever eats into the context).

In [8]:
def build_user_message(context, answer_text):
    return (
        f"Target Answer: {answer_text}\n"
        f"Generate a question from the following context where the target answer is the correct answer. "
        f"Do not include phrases like 'According to the text' in the question and do not repeat the context in the question.\n"
        f"Context: {context}"
    )


def build_chat_prompt_text(context, answer_text):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_message(context, answer_text)},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

## **Generate predictions on the fixed test split**

In [9]:
predictions, references, answers_used = [], [], []

with torch.no_grad():
    for ex in test_ds:
        answer_text = ex["answers"]["text"][0] if ex["answers"]["text"] else ""

        if MODEL_TYPE == "seq2seq":
            prompt = build_prompt(ex["context"], answer_text)   # unchanged, for BART/Flan-T5
            inputs = tokenizer(
                prompt, return_tensors="pt", truncation=True, max_length=MAX_INPUT_LENGTH
            ).to(device)

            output_ids = model.generate(
                **inputs, max_length=MAX_TARGET_LENGTH, num_beams=NUM_BEAMS
            )
            generated = tokenizer.decode(output_ids[0], skip_special_tokens=True)

        else:  # causal LM (Qwen2.5-Instruct) - matches training's manual chat-template tokenization
            prompt_text = build_chat_prompt_text(ex["context"], answer_text)

            input_ids = tokenizer(
                prompt_text, add_special_tokens=False, truncation=True,
                max_length=MAX_INPUT_LENGTH, return_tensors="pt"
            )["input_ids"].to(device)

            output_ids = model.generate(
                input_ids,
                max_new_tokens=MAX_TARGET_LENGTH,
                num_beams=NUM_BEAMS,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.eos_token_id,
            )
            new_tokens = output_ids[0][input_ids.shape[1]:]
            generated = tokenizer.decode(new_tokens, skip_special_tokens=True)

        predictions.append(generated.strip())
        references.append(ex["question"].strip())
        answers_used.append(answer_text)

print(f"Generated {len(predictions)} predictions.")

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Generated 1500 predictions.


## **Reference-based metrics: BLEU, ROUGE-1/2/L, METEOR**

In [10]:
bleu = evaluate.load("sacrebleu")
rouge = evaluate.load("rouge")
meteor = evaluate.load("meteor")

bleu_score = bleu.compute(
    predictions=predictions, references=[[r] for r in references]
)["score"]

rouge_scores = rouge.compute(predictions=predictions, references=references)

meteor_score = meteor.compute(predictions=predictions, references=references)["meteor"]

print(f"BLEU:    {bleu_score:.4f}")
print(f"ROUGE-1: {rouge_scores['rouge1']:.4f}")
print(f"ROUGE-2: {rouge_scores['rouge2']:.4f}")
print(f"ROUGE-L: {rouge_scores['rougeL']:.4f}")
print(f"METEOR:  {meteor_score:.4f}")

[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


BLEU:    10.8312
ROUGE-1: 0.3929
ROUGE-2: 0.1800
ROUGE-L: 0.3490
METEOR:  0.4252


## **Diversity metrics: Distinct-1 / Distinct-2**

Corpus-level, reference-free - flags repetitive/templated question generation (e.g. always producing "What is...?").

In [11]:
def distinct_n(preds, n):
    all_ngrams = []
    for pred in preds:
        tokens = pred.split()
        ngrams = list(zip(*[tokens[i:] for i in range(n)]))
        all_ngrams.extend(ngrams)
    if not all_ngrams:
        return 0.0
    return len(set(all_ngrams)) / len(all_ngrams)


distinct_1 = distinct_n(predictions, 1)
distinct_2 = distinct_n(predictions, 2)

print(f"Distinct-1: {distinct_1:.4f}")
print(f"Distinct-2: {distinct_2:.4f}")

Distinct-1: 0.2613
Distinct-2: 0.6352


## **Fluency metric: GPT-2 perplexity**

Reference-free - measures how natural the generated questions read on their own, independent of closeness to the gold question.

In [12]:
gpt2_tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
gpt2_model = GPT2LMHeadModel.from_pretrained("gpt2").to(device)
gpt2_model.eval()

perplexities = []
with torch.no_grad():
    for text in predictions:
        if not text.strip():
            continue
        encodings = gpt2_tokenizer(text, return_tensors="pt").to(device)
        input_ids = encodings["input_ids"]
        if input_ids.shape[1] < 2:
            continue  # too short to compute a meaningful loss
        outputs = gpt2_model(input_ids, labels=input_ids)
        perplexities.append(torch.exp(outputs.loss).item())

perplexity = float(np.mean(perplexities)) if perplexities else float("nan")
print(f"GPT-2 perplexity: {perplexity:.4f}")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


GPT-2 perplexity: 84.2243


## **Assemble and save results**

In [13]:
results = {
    "model_repo": MODEL_REPO,
    "model_type": MODEL_TYPE,
    "test_size": TEST_SIZE,
    "seed": SEED,
    "bleu": bleu_score,
    "rouge1": rouge_scores["rouge1"],
    "rouge2": rouge_scores["rouge2"],
    "rougeL": rouge_scores["rougeL"],
    "meteor": meteor_score,
    "gpt2_perplexity": perplexity,
    "distinct_1": distinct_1,
    "distinct_2": distinct_2,
}

print("=== Results ===")
for k, v in results.items():
    if isinstance(v, float):
        print(f"{k:20s}: {v:.4f}")
    else:
        print(f"{k:20s}: {v}")

with open(OUTPUT_JSON, "w") as f:
    json.dump(
        {
            "results": results,
            "sample_predictions": [
                {"answer": a, "gold_question": r, "generated_question": p}
                for a, r, p in list(zip(answers_used, references, predictions))[:10]
            ],
        },
        f,
        indent=2,
    )

print(f"\nSaved results + sample predictions to {OUTPUT_JSON}")

=== Results ===
model_repo          : gaurav-dey/llama3.2-3b-qg
model_type          : causal
test_size           : 1500
seed                : 42
bleu                : 10.8312
rouge1              : 0.3929
rouge2              : 0.1800
rougeL              : 0.3490
meteor              : 0.4252
gpt2_perplexity     : 84.2243
distinct_1          : 0.2613
distinct_2          : 0.6352

Saved results + sample predictions to /kaggle/working/eval_results_llama3.2-3b-qg.json


## **Inspect sample generations**

In [14]:
for a, r, p in list(zip(answers_used, references, predictions))[:10]:
    print(f"Answer:              {a}")
    print(f"Gold question:       {r}")
    print(f"Generated question:  {p}")
    print("-" * 80)

Answer:              1852
Gold question:       In what year did Massachusetts first require children to be educated in schools?
Generated question:  In what year did compulsory education begin in Massachusetts?
--------------------------------------------------------------------------------
Answer:              1962
Gold question:       When were stromules discovered?
Generated question:  When were stromules first observed in plant cells?
--------------------------------------------------------------------------------
Answer:              Horace Walpole
Gold question:       Which artist who had a major influence on the Gothic Revival is represented in the V&A's British galleries?
Generated question:  Who was a major influence on the Gothic Revival?
--------------------------------------------------------------------------------
Answer:              several regional colleges and universities
Gold question:       In 1890, who did the university decide to team up with?
Generated question:

## **Next steps**

- Collect each run's `eval_results_*.json` into a single comparison table for the M.Tech project report.
- Optionally extend this notebook with round-trip QA-consistency check, or Item Writing Flaw Pass Rate / Total Structural Flaws once question generation output is paired with generated distractors into full MCQ items.